# 05 — Designing CPs by Composing Operators

The interesting structure of an SRG is determined entirely by the *input* tiling. By pre-composing Conway operators on top of a regular tiling you can quickly explore a wide design space.

This notebook is a **recipe book**: each cell shows one composition, with a one-line description.

In [ ]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')

import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    classifiers,
    colorization,
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    reciprocal_figures,
    rendering,
)
from eucare.rendering import multi_show


In [ ]:
from eucare.search_trees import face_bfs_tree
from eucare.reciprocal_figures import assign_this_way_by_face_z_order, make_SRG


def srg_pipeline(G):
    """Run the standard SRG pipeline: BFS z-order -> SRG -> recompute."""
    central = min(G.faces, key=lambda f: np.linalg.norm(f.midpoint()))
    central['z_order'] = 0
    for orig, dest in face_bfs_tree(central):
        dest['z_order'] = orig['z_order'] + 1
    assign_this_way_by_face_z_order(G)
    SRG = make_SRG(G)
    SRG.recompute_lengths_and_angles()
    return SRG


In [ ]:
def show_pair(G, title):
    """Render a tiling and its SRG side-by-side."""
    SRG = srg_pipeline(G)
    multi_show([G, SRG],
               titles=[f'tiling: {title}', 'SRG'],
               face_inset=0.04, render_vertices=False)


## Visualizing operator fundamental domains

Every `GeometricConwayOperator` is built from a small fundamental-domain graph with three corner vertices `(v1, vf, v2)`. `op.show()` renders that domain, colouring elements by role:

- **orange** — the three triangle corners,
- **red** — vertices/edges marked `delete=True` (removed during substitution),
- **green** — vertices marked `join=True` (collapsed if order-2 after substitution),
- **grey** — retained elements.

Pass `annotate_barycentric=True` to also print each vertex's barycentric coordinates relative to `(v1, vf, v2)`.

In [ ]:
conway.dual_graph().show(filename='conway_dual', annotate_barycentric=True)

In [ ]:
for name in ['kis_graph', 'gyro_graph', 'alternating_flagstone_graph', 'flagstone_pvitelli_graph']:
    print(f'--- {name} ---')
    getattr(conway, name)().show(filename=f'conway_{name}')

## Recipe 1: ambo on hexagons

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(6), rings=2)
G = conway.ambo_graph()(G, delete_on_border=True)
G.recompute_lengths_and_angles()
show_pair(G, 'ambo(hex)')


## Recipe 2: kis on the 3.3.4.3.4 Archimedean tiling

In [ ]:
G = example_graphs.from_tiles(example_tilesets.t_3_3_4_3_4(), rings=2)
G = conway.kis_graph()(G, delete_on_border=True)
G.recompute_lengths_and_angles()
show_pair(G, 'kis(3.3.4.3.4)')


## Recipe 3: alternating-flagstone on a square tiling

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=2)
G = conway.alternating_flagstone_graph()(G, delete_on_border=True)
G.recompute_lengths_and_angles()
show_pair(G, 'alt-flagstone(squares)')


## Recipe 4: Pietro-Vitelli flagstones on hexagons

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(6), rings=2)
G = conway.flagstone_pvitelli_graph()(G, delete_on_border=True)
G.recompute_lengths_and_angles()
show_pair(G, 'flagstone_pvitelli(hex)')


## Going further

- Conway operators compose freely: e.g. `kis(ambo(platonic(6)))`.
- All Archimedean tilings in `example_tilesets` are valid starting points; same for `curved_platonic` (see `02_Curved_Geometries`).
- The legacy notebooks `Intersecting Cylinders.ipynb` and `Winni Leung Corrugations.ipynb` contain more advanced compositions.